In [ ]:
# Archivo original
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

# =========================================================
# CAPTURA DE VARIABLES DE ENTORNO (EL PUENTE CON DOCKER)
# =========================================================

SIMULATION_ENV = os.environ.get("SIMULACION", "True")
SIMULATION_MODE = SIMULATION_ENV.upper() == "TRUE"


AWS_BUCKET = os.environ.get("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "daily_telemetry.csv")
AZURE_CONTAINER = os.environ.get("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")

BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage.")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    print(f"🎛️  Modo de Simulación: {'ACTIVADO (3 filas)' if SIMULATION_MODE else 'DESACTIVADO (1,000 filas con AWS S3)'}")
    
    s3, azure_container_client = None, None
    
    if not SIMULATION_MODE:
        print("⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...")
        from moto import mock_aws
        
        mock_context = mock_aws()
        mock_context.start()
        
        s3 = boto3.client("s3", region_name="us-east-1")
        s3.create_bucket(Bucket=AWS_BUCKET)
        
        headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
        bulk_data = headers
        for i in range(1, 1001):
            code = "P0300" if i % 50 == 0 else "P0171" if i % 75 == 0 else "NONE"
            bulk_data += f"{datetime.now()},VIN-{100000 + i},2100,90,{code}\n"
            
        s3.put_object(Bucket=AWS_BUCKET, Key=AWS_KEY, Body=bulk_data.encode("utf-8"))
        print(f"📦 [S3 Mock] Archivo '{AWS_KEY}' con 1,000 registros cargado con éxito en el Bucket virtual.")
        
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                print(f"☁️  [Local Container] Reporte '{name}' interceptado exitosamente en simulación local.")
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        print("⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.")
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULATION_MODE)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULATION_MODE)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")


In [1]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.github/workflows/main.yml
name: Enterprise Multi-Cloud Telemetry Pipeline - Next 9

on:
  push:
    branches: [ "main" ]
  pull_request:
    branches: [ "main" ]

jobs:
  test-build-and-push:
    runs-on: ubuntu-latest

    defaults:
      run:
        working-directory: containers/ninth_container/repaso/next_9

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

      - name: Run unit tests
        run: |
          pytest test_app.py

      # --- DESPLIEGUE A DOCKER HUB (CD) ---
      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3

      - name: Log in to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Build and Push Docker Image
        uses: docker/build-push-action@v5
        with:
          context: containers/ninth_container/repaso/next_9
          file: containers/ninth_container/repaso/next_9/Dockerfile
          push: true
          tags: ${{ secrets.DOCKERHUB_USERNAME }}/next9-telemetry-batch:latest

Overwriting /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.github/workflows/main.yml


In [2]:
%%writefile /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.gitignore
# Ignorar cuadernos borrador de Jupyter y sus temporales
*.ipynb
.ipynb_checkpoints/
*.ipynb_checkpoints/

# Entornos virtuales y caché de Python
__pycache__/
*.pyc
.pytest_cache/

# Carpetas locales de datos de salida
fleet_data/
data/
.env

Overwriting /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.gitignore


In [3]:
# 1. Agregar la regla de gitignore
!git add /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.gitignore

# 2. Agregar únicamente los archivos de producción de next_9 y el workflow
!git add /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container/repaso/next_9
!git add /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/.github/workflows/main.yml

# 3. Hacer el commit
!git commit -m "feat(next_9): setup streaming multi-cloud enterprise pipeline"

# 4. Enviar a GitHub
!git push origin main

[main 850da38] feat(next_9): setup streaming multi-cloud enterprise pipeline
 6 files changed, 191 insertions(+), 11 deletions(-)
 create mode 100644 containers/ninth_container/repaso/next_9/Dockerfile
 create mode 100644 containers/ninth_container/repaso/next_9/app.py
 create mode 100644 containers/ninth_container/repaso/next_9/requirements.txt
 create mode 100644 containers/ninth_container/repaso/next_9/test_app.py
Counting objects: 14, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (11/11), done.
Writing objects: 100% (14/14), 3.88 KiB | 1.94 MiB/s, done.
Total 14 (delta 5), reused 0 (delta 0)
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   32ec2eb..850da38  main -> main


In [4]:
!docker pull leopinzon75/next9-telemetry-batch:latest
!docker run --rm leopinzon75/next9-telemetry-batch:latest

latest: Pulling from leopinzon75/next9-telemetry-batch

450697fa: Already exists 
55e0dd95: Already exists 
10ea5bd9: Already exists 
e06c2d45: Already exists 
df9cb115: Pulling fs layer 
acbeb0c7: Pulling fs layer 
3433b7d5: Pulling fs layer 
Digest: sha256:d7e5ff3907ac07f9bf402daffe35e9f20ed89494d6d9a7074ad4318e9a2510b9ADownloading  31.87MB/40.95MB
Status: Downloaded newer image for leopinzon75/next9-telemetry-batch:latest
docker.io/leopinzon75/next9-telemetry-batch:latest
🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🎛️  Modo de Simulación: DESACTIVADO (1,000 filas con AWS S3)
⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...
📦 [S3 Mock] Archivo 'daily_telemetry.csv' con 1,000 registros cargado con éxito en el Bucket virtual.
📥 Connecting to S3 Intake Stream...
💾 [Auditoría Local] Réplica cruda de entrada guardada en: data/archive/input/raw_s3_mirror_20260731_190104.csv
📤 Preparing handoff for 27 processed records...
☁️  [Local Container] Reporte 

In [6]:
%%writefile app.py
import io
import os
from pathlib import Path
from datetime import datetime
import boto3
from botocore.config import Config
from azure.storage.blob import BlobServiceClient

# =========================================================
# CAPTURA DE VARIABLES DE ENTORNO (EL PUENTE CON DOCKER)
# =========================================================
# Si SIMULACION no viene configurada, por defecto será False (Modo Enterprise / 1000 filas)
SIMULATION_MODE = os.getenv("SIMULACION", "False").lower() == "true"

AWS_BUCKET = os.environ.get("AWS_INPUT_BUCKET", "dev-fleet-raw-data")
AWS_KEY = os.environ.get("AWS_INPUT_KEY", "daily_telemetry.csv")
AZURE_CONTAINER = os.environ.get("AZURE_OUTPUT_CONTAINER", "dev-fleet-alerts")

BASE_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
LOCAL_INPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "input"
LOCAL_OUTPUT_ARCHIVE = BASE_DIR / "data" / "archive" / "output"

def s3_telemetry_streamer(s3_client, bucket, key, simulation_mode=False):
    if simulation_mode:
        print("🛠️  [Simulación] Fabricando flujo de datos local en memoria (3 filas)...")
        s3_data = (
            "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
            f"{datetime.now()},VIN-999111,2500,95,P0300\n"
            f"{datetime.now()},VIN-222333,1800,88,NONE\n"
            f"{datetime.now()},VIN-444555,3100,102,P0171\n"
        )
    else:
        response = s3_client.get_object(Bucket=bucket, Key=key)
        s3_data = response["Body"].read().decode("utf-8")
    
    LOCAL_INPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_raw_file = LOCAL_INPUT_ARCHIVE / f"raw_s3_mirror_{timestamp_str}.csv"
    local_raw_file.write_text(s3_data, encoding="utf-8")
    print(f"💾 [Auditoría Local] Réplica cruda de entrada guardada en: {local_raw_file.relative_to(BASE_DIR)}")

    stream_buffer = io.StringIO(s3_data)
    stream_buffer.readline()
    
    for line in stream_buffer:
        if line.strip():
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def azure_bulk_writer(container_client, blob_name, processed_records, simulation_mode=False):
    if not processed_records:
        return
    
    headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n"
    csv_buffer = headers
    for record in processed_records:
        csv_buffer += (
            f"{record['timestamp']},{record['vehicle_id']},{record['engine_rpm']},"
            f"{record['coolant_temp_c']},{record['fault_code']},{record['alert_status']},"
            f"{record['processed_at']}\n"
        )
    
    if simulation_mode:
        print("☁️  [Simulación] Modo local activo. Se saltó la carga real a Azure.")
    else:
        container_client.upload_blob(name=blob_name, data=csv_buffer, overwrite=True)
        print("☁️  [Cloud] Transmisión exitosa a Azure Blob Storage (MOCK/Real).")
    
    LOCAL_OUTPUT_ARCHIVE.mkdir(parents=True, exist_ok=True)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_clean_file = LOCAL_OUTPUT_ARCHIVE / f"clean_alerts_{timestamp_str}.csv"
    local_clean_file.write_text(csv_buffer, encoding="utf-8")
    print(f"💾 [Auditoría Local] Reporte filtrado guardado en: {local_clean_file.relative_to(BASE_DIR)}")

if __name__ == '__main__':
    print("🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...")
    print(f"🎛️  Modo de Simulación (3 filas): {'ACTIVADO' if SIMULATION_MODE else 'DESACTIVADO (Modo Enterprise: 1,000 filas con AWS S3)'}")
    
    s3, azure_container_client = None, None
    
    if not SIMULATION_MODE:
        print("⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...")
        from moto import mock_aws
        
        mock_context = mock_aws()
        mock_context.start()
        
        s3 = boto3.client("s3", region_name="us-east-1")
        s3.create_bucket(Bucket=AWS_BUCKET)
        
        headers = "timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n"
        bulk_data = headers
        for i in range(1, 1001):
            code = "P0300" if i % 50 == 0 else "P0171" if i % 75 == 0 else "NONE"
            bulk_data += f"{datetime.now()},VIN-{100000 + i},2100,90,{code}\n"
            
        s3.put_object(Bucket=AWS_BUCKET, Key=AWS_KEY, Body=bulk_data.encode("utf-8"))
        print(f"📦 [S3 Mock] Archivo '{AWS_KEY}' con 1,000 registros cargado con éxito en el Bucket virtual.")
        
        class LocalMockAzureContainer:
            def upload_blob(self, name, data, overwrite=True):
                print(f"☁️  [Local Container] Reporte '{name}' interceptado exitosamente en simulación local.")
        
        azure_container_client = LocalMockAzureContainer()
        
    else:
        print("⚠️  Advertencia: Modo de Simulación de 3 filas en memoria Activo.")
    
    print("📥 Connecting to S3 Intake Stream...")
    data_stream = s3_telemetry_streamer(s3, AWS_BUCKET, AWS_KEY, simulation_mode=SIMULATION_MODE)
    
    incidents = []
    for raw_row in data_stream:
        clean_row = telemetry_processor(raw_row)
        if clean_row:
            incidents.append(clean_row)
            
    print(f"📤 Preparing handoff for {len(incidents)} processed records...")
    azure_bulk_writer(azure_container_client, "critical_incidents_report.csv", incidents, simulation_mode=SIMULATION_MODE)
    
    print("🏁 Batch complete! Multi-cloud architectural loop completed successfully.")

Overwriting app.py


In [8]:
# 1. Agrega el código corregido
!git add app.py

# 2. Crea el commit
!git commit -m "fix(next_9): ajustar SIMULATION_MODE a False para pruebas con 1000 registros"

# 3. Dispara la integración continua hacia GitHub Actions y Docker Hub
!git push origin main

[main 1208ccd] fix(next_9): ajustar SIMULATION_MODE a False para pruebas con 1000 registros
 1 file changed, 5 insertions(+), 6 deletions(-)
Counting objects: 7, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 813 bytes | 813.00 KiB/s, done.
Total 7 (delta 6), reused 0 (delta 0)
remote: Resolving deltas: 100% (6/6), completed with 6 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   850da38..1208ccd  main -> main


In [9]:
!docker pull leopinzon75/next9-telemetry-batch:latest

latest: Pulling from leopinzon75/next9-telemetry-batch

450697fa: Already exists 
55e0dd95: Already exists 
10ea5bd9: Already exists 
e06c2d45: Already exists 
27d199c5: Pulling fs layer 
405a7c82: Pulling fs layer 
a42bc5f8: Pulling fs layer 
46c5a87f: Pull complete 316kB/7.316kBBADownloading  28.17MB/40.95MBExtracting  10.65MB/40.95MBDigest: sha256:d07a3239f23f3267bf68a299149330a079ebc6d9228dd713aa52dc68714b09ec
Status: Downloaded newer image for leopinzon75/next9-telemetry-batch:latest
docker.io/leopinzon75/next9-telemetry-batch:latest


In [10]:
!docker run --rm leopinzon75/next9-telemetry-batch:latest

🚀 Live Enterprise Multi-Cloud Pipeline Booting Up...
🎛️  Modo de Simulación (3 filas): DESACTIVADO (Modo Enterprise: 1,000 filas con AWS S3)
⚡ [Modo Enterprise Local] Inicializando entorno simulado de AWS S3 con Moto...
📦 [S3 Mock] Archivo 'daily_telemetry.csv' con 1,000 registros cargado con éxito en el Bucket virtual.
📥 Connecting to S3 Intake Stream...
💾 [Auditoría Local] Réplica cruda de entrada guardada en: data/archive/input/raw_s3_mirror_20260731_194305.csv
📤 Preparing handoff for 27 processed records...
☁️  [Local Container] Reporte 'critical_incidents_report.csv' interceptado exitosamente en simulación local.
☁️  [Cloud] Transmisión exitosa a Azure Blob Storage (MOCK/Real).
💾 [Auditoría Local] Reporte filtrado guardado en: data/archive/output/clean_alerts_20260731_194305.csv
🏁 Batch complete! Multi-cloud architectural loop completed successfully.


In [12]:
!head -n 5 data/archive/output/clean_alerts_*.csv

timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at
2026-07-31 12:31:57.829327,VIN-100050,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848863
2026-07-31 12:31:57.829408,VIN-100075,2100,90,P0171,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848932
2026-07-31 12:31:57.829490,VIN-100100,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849005
2026-07-31 12:31:57.829660,VIN-100150,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849080


In [13]:
!cat data/archive/output/clean_alerts_*.csv

timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at
2026-07-31 12:31:57.829327,VIN-100050,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848863
2026-07-31 12:31:57.829408,VIN-100075,2100,90,P0171,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848932
2026-07-31 12:31:57.829490,VIN-100100,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849005
2026-07-31 12:31:57.829660,VIN-100150,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849080
2026-07-31 12:31:57.829837,VIN-100200,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849146
2026-07-31 12:31:57.829930,VIN-100225,2100,90,P0171,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849204
2026-07-31 12:31:57.830021,VIN-100250,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849242
2026-07-31 12:31:57.830259,VIN-100300,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849306
2026-07-31 12:31:57.830487,VIN-10035

In [14]:
!less data/archive/output/clean_alerts_*.csv

timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at
2026-07-31 12:31:57.829327,VIN-100050,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848863
2026-07-31 12:31:57.829408,VIN-100075,2100,90,P0171,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.848932
2026-07-31 12:31:57.829490,VIN-100100,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849005
2026-07-31 12:31:57.829660,VIN-100150,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849080
2026-07-31 12:31:57.829837,VIN-100200,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849146
2026-07-31 12:31:57.829930,VIN-100225,2100,90,P0171,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849204
2026-07-31 12:31:57.830021,VIN-100250,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849242
2026-07-31 12:31:57.830259,VIN-100300,2100,90,P0300,⚠️ CRITICAL DEVIATION DETECTED,2026-07-31 12:31:57.849306
2026-07-31 12:31:57.830487,VIN-10035

In [15]:
!column -s, -t < data/archive/output/clean_alerts_*.csv | less -S

timestamp                   vehicle_id  engine_rpm  coolant_temp_c  fault_code  
2026-07-31 12:31:57.829327  VIN-100050  2100        90              P0300       
2026-07-31 12:31:57.829408  VIN-100075  2100        90              P0171       
2026-07-31 12:31:57.829490  VIN-100100  2100        90              P0300       
2026-07-31 12:31:57.829660  VIN-100150  2100        90              P0300       
2026-07-31 12:31:57.829837  VIN-100200  2100        90              P0300       
2026-07-31 12:31:57.829930  VIN-100225  2100        90              P0171       
2026-07-31 12:31:57.830021  VIN-100250  2100        90              P0300       
2026-07-31 12:31:57.830259  VIN-100300  2100        90              P0300       
2026-07-31 12:31:57.830487  VIN-100350  2100        90              P0300       
2026-07-31 12:31:57.830633  VIN-100375  2100        90              P0171       
2026-07-31 12:31:57.830738  VIN-100400  2100        90              P0300       
2026-07-31 12:31:57.830946  

In [16]:
!wc -l data/archive/output/clean_alerts_*.csv

      28 data/archive/output/clean_alerts_20260731_123157.csv


In [17]:
!mkdir -p next_10/data/archive/input next_10/data/archive/output